In [0]:
%run ../00.common/1.environment_config

In [0]:
catalog_name


In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.circuits"
silver_table=f"{catalog_name}.{silver_schema}.circuits"

In [0]:
circuits_df=spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F
circuits_selected_df=circuits_df.select(
                        F.col("circuitid"),
                        F.col("circuitName"),
                        F.col("lat"),
                        F.col("long"),
                        F.col("locality"),
                        F.col("country"),
                        F.col("ingestion_timestamp"),
                        F.col("source_file")
)

In [0]:
circuits_renamed_df=(circuits_selected_df
                     .withColumnRenamed("circuitId", "circuit_id")
                     .withColumnRenamed("circuitName", "circuit_name")
                     .withColumnRenamed("lat", "lattittude")
                     .withColumnRenamed("long", "longittude")
                    )
                     

In [0]:
spark.version

In [0]:
from pyspark.sql import functions as F 
circuits_valid_df=circuits_renamed_df.filter(F.col("circuit_id").isNotNull())

In [0]:
#circuits_valid_df=circuits_renamed_df.filter("circuit_id IS NOT NULL")

In [0]:
circuit_distinct_df=circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
circuit_final_df=(circuit_distinct_df
                  .withColumn("circuit_name",F.initcap(F.col("circuit_name")))
                  .withColumn("locality",F.initcap(F.col("locality")))
                  )

In [0]:
(
    circuit_final_df
                 .write
                 .format("delta")
                 .mode("overwrite")
                 .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))